# Лабораторная работа №5. ARP: как IP превращается в MAC

В Lab 003 мы изучили Ethernet destination MAC, а в Lab 004 — FDB коммутатора. Теперь выясним, откуда Linux-host узнаёт MAC для конкретного IPv4 next-hop, и почему **ARP может появиться даже во время повторного ping без очистки cache**.

Эксперимент: первый ARP Request → Reply → ICMP; быстрый повтор с сохранённой записью; затем состояние `STALE` и проверка доступности соседа (NUD). Сначала предсказываем, затем наблюдаем реальные пакеты — без утверждения «при повторном ping ARP гарантированно не будет».

## Схема

![ARP](assets/arp.svg)

`pc1` и `pc2` находятся в одной IPv4 `/24`, между ними L2 switch. Чтобы отправить IPv4 packet локальному соседу, `pc1` сначала нужен destination MAC.

## 1. Настроим адреса

В terminal `pc1`:

```bash
ip link set eth1 up
ip -4 addr flush dev eth1
ip addr add 10.5.0.10/24 dev eth1
```

В terminal `pc2`:

```bash
ip link set eth1 up
ip -4 addr flush dev eth1
ip addr add 10.5.0.20/24 dev eth1
```

В terminal `pc1` очищаем динамические neighbor-записи перед **первым** экспериментом:

```bash
ip neigh flush dev eth1
ip neigh show 10.5.0.20 dev eth1
```

`eth0` — management plane платформы; его не трогаем.

### Q1. Какой Ethernet destination нужен ARP Request и почему Request нельзя отправить unicast на MAC `pc2`?

`введите ваш ответ`

<details>
<summary>AI REVIEW · проверьте рассуждение</summary>

**AI-REVIEW-PROMPT / REVIEW-ONLY**

Проверь мой ответ на вопрос Q1, расположенный непосредственно выше. Проверь правильность логики и используемых понятий ARP, neighbor cache и Linux NUD. Не раскрывай готовый ответ, не переписывай мой текст и не выдавай команды будущих упражнений. Укажи возможное упущение и задай 1–2 наводящих вопроса.

</details>

## 2. Первый контакт: ARP → ICMP

Запустите capture ниже. Пока он работает, в terminal `pc1` выполните:

```bash
ping -c 2 -W 1 10.5.0.20
```

Сравните ARP Request (обычно Ethernet broadcast), ARP Reply и ICMP Echo. Capture на `pc1:eth1` может показывать ARP и от другой стороны, поэтому проверяйте **sender IP / target IP**, а не только число ARP-пакетов.

In [ ]:
%%capture_traffic pc1:eth1 --filter "arp or icmp" --timeout 12 --packets 100 --save captures/arp-icmp.pcap
import time
print("В terminal pc1: ping -c 2 -W 1 10.5.0.20")
time.sleep(10)


In [ ]:
from cms_labs_jupyter.traffic_capture import PcapViewer
from IPython.display import display
viewer = PcapViewer("captures/arp-icmp.pcap", "arp or icmp", limit=30)
frames = viewer.packets()
display(frames)
for proto in ("ARP", "ICMP"):
    match = frames[frames["Protocol"].astype(str).str.contains(proto, case=False, na=False)]
    if not match.empty:
        print("\n" + proto + "\n")
        print(viewer.packet(int(match.iloc[0]["№"]), "protocols"))

### После первого ping: запись существует, но у неё есть состояние

На `pc1`:

```bash
ip -s neigh show 10.5.0.20 dev eth1
```

Найдите `lladdr` и состояние записи. `REACHABLE` означает, что доступность подтверждена недавно; `STALE` — что MAC известен, но недавнего подтверждения уже нет. `DELAY` и `PROBE` отражают этапы проверки этой записи. Отсутствие записи или `INCOMPLETE` — другой случай: адрес ещё предстоит разрешить.

**Это две разные вещи:** запись `IP → MAC` присутствует в таблице, и ядро считает адресата недавно подтверждённым.

### Q2. Почему ARP Reply уже может быть unicast, хотя Request был broadcast?

`введите ваш ответ`

<details>
<summary>AI REVIEW · проверьте рассуждение</summary>

**AI-REVIEW-PROMPT / REVIEW-ONLY**

Проверь мой ответ на вопрос Q2, расположенный непосредственно выше. Проверь правильность логики и используемых понятий ARP, neighbor cache и Linux NUD. Не раскрывай готовый ответ, не переписывай мой текст и не выдавай команды будущих упражнений. Укажи возможное упущение и задай 1–2 наводящих вопроса.

</details>

## 3. Почему ARP может повториться, даже если MAC уже известен

Linux использует **Neighbor Unreachability Detection (NUD)**, а не «вечный ARP-кеш с одним TTL».

- `REACHABLE`: MAC известен, достижимость недавно подтверждена. Немедленное повторное разрешение адреса обычно не требуется.
- `STALE`: MAC **ещё хранится**, и пакет можно отправить, используя его, но после использования ядро может перейти в `DELAY`.
- `DELAY → PROBE`: если нового подтверждения нет, после задержки могут отправляться **ARP probes**, часто unicast к уже известному MAC. Это не обязательно новый broadcast для поиска адреса.
- `FAILED/INCOMPLETE`: сосед не подтвердился или разрешение ещё идёт; возможны новые ARP Request.

Посмотрите настройки **своего интерфейса** в `pc1`:

```bash
cat /proc/sys/net/ipv4/neigh/eth1/base_reachable_time_ms
cat /proc/sys/net/ipv4/neigh/eth1/delay_first_probe_time
cat /proc/sys/net/ipv4/neigh/eth1/gc_stale_time
ip -s neigh show 10.5.0.20 dev eth1
```

Обычно `base_reachable_time_ms` равен 30000 мс, но реальное время `REACHABLE` рандомизируется примерно от **0,5 до 1,5 базового** и может продлеваться подтверждениями. `delay_first_probe_time` по умолчанию обычно 5 секунд. А `gc_stale_time` задаёт периодичность проверки старых записей сборщиком, **не точный срок удаления и не TTL MAC**.

Параметры доступны для настройки Linux в отдельном namespace, но в этой лабораторной **не меняйте их для прохождения CHECK**: значения могут отличаться в стендах, а доступ к sysctl зависит от контейнера. Вместо изменения таймера ниже мы переведём **одну запись** в `STALE` и посмотрим на эффект. См. [Linux arp(7)](https://man7.org/linux/man-pages/man7/arp.7.html) и [ip-neighbour(8)](https://www.man7.org/linux/man-pages/man8/ip-neighbour.8.html).

### Q3. При повторном ping без `ip neigh flush` запись `10.5.0.20 → MAC` всё ещё существует. Гарантирует ли это отсутствие ARP Request в capture? Чем случаи `REACHABLE` и `STALE` отличаются?

`введите ваш ответ`

<details>
<summary>AI REVIEW · проверьте рассуждение</summary>

**AI-REVIEW-PROMPT / REVIEW-ONLY**

Проверь мой ответ на вопрос Q3, расположенный непосредственно выше. Проверь правильность логики и используемых понятий ARP, neighbor cache и Linux NUD. Не раскрывай готовый ответ, не переписывай мой текст и не выдавай команды будущих упражнений. Укажи возможное упущение и задай 1–2 наводящих вопроса.

</details>

## 4. Второй ping без очистки: сравниваем наблюдения

**Не выполняйте** `ip neigh flush`. Прямо перед запуском capture посмотрите состояние:

```bash
ip -s neigh show 10.5.0.20 dev eth1
```

Запустите capture, затем в terminal `pc1`:

```bash
ping -c 2 -W 1 10.5.0.20
ip -s neigh show 10.5.0.20 dev eth1
```

Сравните с первым PCAP. **Нуль ARP — возможный ожидаемый результат, но не обязательное условие успеха.** Если ARP появился, проверьте, кто его отправил, какой MAC/IPv4 он выясняет и каким было состояние neighbor-записи. Это может быть проверка `STALE`, другая сторона соединения или иной ARP-трафик.

In [ ]:
%%capture_traffic pc1:eth1 --filter "arp or icmp" --timeout 12 --packets 100 --save captures/arp-repeat.pcap
import time
print("Без flush! На pc1: ping -c 2 -W 1 10.5.0.20")
time.sleep(10)


In [ ]:
from cms_labs_jupyter.traffic_capture import PcapViewer
from IPython.display import display

repeat = PcapViewer("captures/arp-repeat.pcap", "arp or icmp", limit=50)
repeat_frames = repeat.packets()
display(repeat_frames)
arps = repeat_frames[repeat_frames["Protocol"].astype(str).str.contains("ARP", case=False, na=False)]
print(f"ARP frames в повторном capture: {len(arps)} — это не обязательно повторное разрешение от pc1")
for number in arps["№"].head(5):
    print(f"\nARP frame {int(number)}: проверьте sender/target IP и Ethernet destination")
    print("\n".join(repeat.packet(int(number), "protocols").splitlines()[:45]))


## 5. Управляемый эксперимент: MAC известен, но запись `STALE`

Не будем угадывать момент естественного устаревания. На `pc1` сохраним уже изученный MAC соседа и установим для **этой записи** состояние `STALE`:

```bash
MAC=$(ip neigh show 10.5.0.20 dev eth1 | awk '{for(i=1;i<=NF;i++) if($i=="lladdr") {print $(i+1); exit}}')
test -n "$MAC" && ip neigh replace 10.5.0.20 lladdr "$MAC" nud stale dev eth1
ip -s neigh show 10.5.0.20 dev eth1
```

Если `MAC` оказался пустым, сначала повторите обычный ping и получите neighbor-запись; затем вернитесь к командам выше. **Это намеренно меняет состояние cache, а не конфигурацию адреса или порта.**

### Q4. После перевода записи в `STALE` её `lladdr` сохранён. Почему первый ICMP может успешно дойти без нового ARP broadcast, но некоторое время спустя Linux всё равно может отправить ARP probe? В чём здесь роль `delay_first_probe_time`?

`введите ваш ответ`

<details>
<summary>AI REVIEW · проверьте рассуждение</summary>

**AI-REVIEW-PROMPT / REVIEW-ONLY**

Проверь мой ответ на вопрос Q4, расположенный непосредственно выше. Проверь правильность логики и используемых понятий ARP, neighbor cache и Linux NUD. Не раскрывай готовый ответ, не переписывай мой текст и не выдавай команды будущих упражнений. Укажи возможное упущение и задай 1–2 наводящих вопроса.

</details>

### Наблюдаем NUD, а не просто повторяем очистку cache

Откройте terminal `pc1`, приготовьте команду ниже, затем запустите capture. Пока он работает, выполните:

```bash
ping -c 10 -i 1 -W 1 10.5.0.20
ip -s neigh show 10.5.0.20 dev eth1
```

Захват длиннее предыдущего, чтобы дать ядру время перейти от `DELAY` к возможному `PROBE`. Если ядро подтвердит доступность другим способом, ARP probe может и не появиться. Мы сравниваем **состояние cache, таймеры и реальные кадры**, а не добиваемся искусственно обязательного ARP.

In [ ]:
%%capture_traffic pc1:eth1 --filter "arp or icmp" --timeout 17 --packets 100 --save captures/arp-stale.pcap
import time
print("Пока идёт capture, на pc1: ping -c 10 -i 1 -W 1 10.5.0.20")
time.sleep(15)


In [ ]:
from cms_labs_jupyter.traffic_capture import PcapViewer
from IPython.display import display

stale = PcapViewer("captures/arp-stale.pcap", "arp or icmp", limit=60)
stale_frames = stale.packets()
display(stale_frames)
stale_arps = stale_frames[stale_frames["Protocol"].astype(str).str.contains("ARP", case=False, na=False)]
print(f"ARP frames при использовании STALE entry: {len(stale_arps)}")
for number in stale_arps["№"].head(6):
    print(f"\nARP frame {int(number)}")
    print("\n".join(stale.packet(int(number), "protocols").splitlines()[:45]))


### Как интерпретировать результаты

Сопоставьте три наблюдения: **`ip -s neigh` до ping**, **ARP sender/target в PCAP**, **`ip -s neigh` после ping**.

ARP Request, направленный на известный MAC, может быть **unicast NUD probe**, а не первым broadcast-обнаружением. Наличие записи в `ip neigh` не гарантирует, что ядро больше никогда не будет проверять доступность. И наоборот, если повторный capture не содержит ARP, это не означает, что cache «вечный»: возможно, запись была `REACHABLE` или доступность уже получила подтверждение.

**Не оценивайте эксперимент только по числу ARP.** Объяснение должно учитывать направление и назначение ARP, neighbor state и реальные параметры ядра.

## CHECK

Верните рабочее состояние обычным ping, затем проверьте на `pc1`:

```bash
ping -c 2 -W 1 10.5.0.20
ip neigh show 10.5.0.20 dev eth1
```

Checker оценивает адреса, связность и наличие `lladdr` в neighbor table на `pc1`. Он **не требует нуля ARP во втором capture и не оценивает конкретное состояние NUD**: `REACHABLE`, `STALE`, `DELAY` могут быть нормальными в разные моменты.

## Итог

- IPv4 route определяет next-hop, ARP сопоставляет его IPv4 с MAC для локального L2-сегмента.
- Первый контакт после очистки записи обычно создаёт ARP Request/Reply, затем возможен ICMP.
- Кеширование IP→MAC и **подтверждение доступности соседа — не одно и то же**.
- Повторный ping **может** обойтись без ARP, но `STALE → DELAY → PROBE` позволяет Linux проверять даже существующую запись.
- `base_reachable_time_ms` определяет базу рандомизированного срока `REACHABLE`; `delay_first_probe_time` — задержку перед проверками; `gc_stale_time` **не TTL каждой ARP-записи**.
- Чтобы понять появившийся ARP, нужно видеть **sender/target IP, destination MAC и neighbor state**, а не только количество ARP packets.

Следующая лабораторная расширит наблюдение за взаимодействием узлов на сети.